# Control Variables - 12 Oil
29/06/2026, Kuba Kowalski 

Definition: percentage of province area that overlaps with shapefile of petroleum extraction. 

In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.neighbors import BallTree
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import gaussian_kde

# ------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------

province_polygons = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_residence_all_with_cote.geojson"
)


oil_file = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\ancilliary_data\12_oil\PETRODATA v12 Data\Petrodata_Onshore_V1.2.shp"
)

oil_out_dir = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\12_oil"
)
oil_out_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------------
zone_id = "GEOLEVEL1"
oil_var = "12_oil-surface-share"
area_crs = "EPSG:6933"

In [2]:
# ------------------------------------------------------------------
# PREPARE DATA
# ------------------------------------------------------------------
provinces = gpd.read_file(province_polygons)
provinces_oil = provinces.copy()
oil = gpd.read_file(oil_file)

provinces_oil[zone_id] = (
    provinces_oil[zone_id]
    .astype(str)
    .str.strip()
    .str.zfill(6)
)

provinces_oil["geometry"] = provinces_oil.geometry.make_valid()
oil["geometry"] = oil.geometry.make_valid()

provinces_oil = provinces_oil[
    provinces_oil.geometry.notna() &
    ~provinces_oil.geometry.is_empty
].copy()

oil = oil[
    oil.geometry.notna() &
    ~oil.geometry.is_empty
].copy()

provinces_unique_oil = provinces_oil.dissolve(
    by=zone_id,
    as_index=False
)

print("Oil features retained:", len(oil))
print("Oil geometry types:")
print(oil.geom_type.value_counts(dropna=False))

# ------------------------------------------------------------------
# PROJECT TO AREA CRS
# ------------------------------------------------------------------

if oil.crs is None:
    raise ValueError("Oil shapefile has no CRS.")

if provinces_unique_oil.crs is None:
    raise ValueError("Province file has no CRS.")

provinces_m = provinces_unique_oil.to_crs(area_crs)
oil_m = oil.to_crs(area_crs)

provinces_m["geometry"] = provinces_m.geometry.make_valid()
oil_m["geometry"] = oil_m.geometry.make_valid()

# Dissolve oil polygons to avoid double-counting overlapping fields
oil_union = gpd.GeoDataFrame(
    {"oil_id": [1]},
    geometry=[oil_m.geometry.union_all()],
    crs=area_crs
)

provinces_m["province_area_m2"] = provinces_m.geometry.area


Oil features retained: 891
Oil geometry types:
Polygon         888
MultiPolygon      3
Name: count, dtype: int64


In [3]:
# ------------------------------------------------------------------
# INTERSECT PROVINCES WITH OIL FIELDS
# ------------------------------------------------------------------

intersection = gpd.overlay(
    provinces_m[[zone_id, "province_area_m2", "geometry"]],
    oil_union[["oil_id", "geometry"]],
    how="intersection",
    keep_geom_type=True
)

if len(intersection) > 0:
    intersection["oil_area_m2"] = intersection.geometry.area

    oil_area_by_province = (
        intersection
        .groupby(zone_id)["oil_area_m2"]
        .sum()
        .reset_index()
    )

else:
    oil_area_by_province = pd.DataFrame(
        columns=[zone_id, "oil_area_m2"]
    )

# ------------------------------------------------------------------
# COMPLETE OUTPUT FOR ALL PROVINCES
# ------------------------------------------------------------------

oil_df = (
    provinces_m[[zone_id, "province_area_m2"]]
    .merge(oil_area_by_province, on=zone_id, how="left")
)

oil_df["oil_area_m2"] = oil_df["oil_area_m2"].fillna(0)

oil_df[oil_var] = (
    oil_df["oil_area_m2"] /
    oil_df["province_area_m2"]
)

oil_df[oil_var] = oil_df[oil_var].clip(0, 1)

oil_df = oil_df[
    [zone_id, oil_var]
].copy()

# ------------------------------------------------------------------
# JOIN BACK TO SPATIAL FILE FOR INSPECTION
# ------------------------------------------------------------------

provinces_oil_output = provinces_oil.merge(
    oil_df,
    on=zone_id,
    how="left"
)

In [4]:
# ------------------------------------------------------------------
# EXPORT
# ------------------------------------------------------------------

csv_path = oil_out_dir / "12_oil_surface_share.csv"
gpkg_path = oil_out_dir / "12_oil_surface_share.gpkg"

oil_df.to_csv(csv_path, index=False)
provinces_oil_output.to_file(gpkg_path, driver="GPKG")

print(f"\nCSV saved to: {csv_path}")
print(f"GPKG saved to: {gpkg_path}")

# ------------------------------------------------------------------
# SANITY CHECKS
# ------------------------------------------------------------------

print("\nOil surface share summary:")
print(oil_df[oil_var].describe())

print("\nProvinces with oil share > 0:")
print((oil_df[oil_var] > 0).sum())

print("\nPreview:")
print(oil_df.head())

print("Done.")


CSV saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\12_oil\12_oil_surface_share.csv
GPKG saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\12_oil\12_oil_surface_share.gpkg

Oil surface share summary:
count    304.000000
mean       0.006218
std        0.059114
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        0.959806
Name: 12_oil-surface-share, dtype: float64

Provinces with oil share > 0:
14

Preview:
  GEOLEVEL1  12_oil-surface-share
0    000001                   0.0
1    000002                   0.0
2    000003                   0.0
3    000004                   0.0
4    000005                   0.0
Done.


In [5]:
# ------------------------------------------------------------------
# LOAD OUTPUTS IF NEEDED
# ------------------------------------------------------------------


africa_outline_file = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\IPUMS_boundaries\world_countries_africa_only.json"
)
oil_gpkg = oil_out_dir / "12_oil_surface_share.gpkg"

if "provinces_oil_output" not in globals():
    provinces_oil_output = gpd.read_file(oil_gpkg)

if "africa" not in globals():
    africa = gpd.read_file(africa_outline_file)

if africa.crs != provinces_oil_output.crs:
    africa = africa.to_crs(provinces_oil_output.crs)

map_output_dir = oil_out_dir / "maps"
map_output_dir.mkdir(exist_ok=True)

# Avoid duplicate cohort geometries in visualization
provinces_plot_base = (
    provinces_oil_output
    .sort_values(zone_id)
    .drop_duplicates(subset=[zone_id])
    .copy()
)

# ------------------------------------------------------------------
# MAP
# ------------------------------------------------------------------

var = "12_oil-surface-share"
label = "Share of province surface area containing oil fields"

gdf_plot = provinces_plot_base[provinces_plot_base[var].notna()].copy()
values = gdf_plot[var].dropna().values

if len(values) == 0:
    print(f"Skipping {var}: no valid values")

else:
    vmin = 0
    vmax = max(values.max(), 0.01)

    fig, ax = plt.subplots(figsize=(16, 20))

    africa.plot(
        ax=ax,
        facecolor="none",
        edgecolor="lightgrey",
        linewidth=0.5
    )

    gdf_plot.plot(
        column=var,
        cmap="viridis",
        linewidth=0.4,
        edgecolor="black",
        legend=False,
        vmin=vmin,
        vmax=vmax,
        ax=ax
    )

    ax.set_title(label)
    ax.set_axis_off()

    minx, miny, maxx, maxy = africa.total_bounds
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap="viridis")
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=ax,
        orientation="horizontal",
        fraction=0.05,
        pad=0.04
    )

    cbar.set_label("Share of province area")

    cb_pos = cbar.ax.get_position()

    wave_ax = fig.add_axes([
        cb_pos.x0,
        cb_pos.y1 + 0.005,
        cb_pos.width,
        0.05
    ])

    mean_val = values.mean()
    min_val = values.min()
    max_val = values.max()

    fig.text(
        0.5,
        0.02,
        f"Mean: {mean_val:.3f}   Min: {min_val:.3f}   Max: {max_val:.3f}",
        ha="center",
        fontsize=10
    )

    if len(values) > 1 and values.min() < values.max():
        kde = gaussian_kde(values)
        x = np.linspace(vmin, vmax, 300)
        y = kde(x)

        wave_ax.plot(x, y, linewidth=1.5)
        wave_ax.fill_between(x, y, alpha=0.25)

    else:
        constant_val = values[0] if len(values) > 0 else np.nan

        if np.isfinite(constant_val):
            wave_ax.axvline(
                constant_val,
                linewidth=1.5,
                linestyle="--"
            )

            wave_ax.text(
                constant_val,
                0.5,
                "constant",
                ha="center",
                va="center",
                fontsize=8,
                transform=wave_ax.get_xaxis_transform()
            )

    wave_ax.set_xlim(vmin, vmax)
    wave_ax.set_xticks([])
    wave_ax.set_yticks([])

    for spine in wave_ax.spines.values():
        spine.set_visible(False)

    out_file = map_output_dir / "12_oil_surface_share_map.png"

    plt.savefig(
        out_file,
        dpi=800,
        bbox_inches="tight"
    )

    plt.close()

    print(f"Saved: {out_file}")

print("Done.")

Saved: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\12_oil\maps\12_oil_surface_share_map.png
Done.
